# AkidaNet model prep: load, convert, check on the AKD1000

Prepares the model used in `Phase 1 - No AI/akidanet-imagenetc-card.ipynb` and `-sim.ipynb`. **No training.**

1. Load BrainChip's pretrained, quantized AkidaNet: ImageNet, alpha 0.5, 224x224, 8-bit first layer, 4-bit weights and activations. The first call downloads the weights from BrainChip. This is the CNN baseline.
2. Convert it with `cnn2snn` to an Akida 1.0 model and save `akidanet_imagenet_a05_224_v1.fbz`. This is the SNN.
3. Map it on the AKD1000 and check it on BrainChip's 10 sample images. Compare Keras vs card and `predict` vs `forward`, and read the power.

Steps 1-2 run anywhere. Step 3 needs the card and is skipped without one. That keeps simulator numbers out. No dataset is needed.

**Why AkidaNet:** the AKD1000 is Akida 1.0. MobileNetV2 does not convert for it: Keras MobileNet's depthwise blocks are invalid on Akida 1.0, which expects separable convolutions. AkidaNet is BrainChip's own Akida 1.0 network and comes pretrained and quantized.

**Class scores: `predict`, not `forward`.** `forward` returns the raw int32 potentials of the last layer. `predict` returns them "obtained from the model discrete potentials by applying a shift and a scale" (akida API docs). The classifier has a per-class `act_step`, so the argmax of `forward` is not the class.

Versions used: Python 3.11, akida 2.19.3, cnn2snn 2.19.3, quantizeml 1.2.5, akida_models 1.14.2, TF 2.19.1 + tf_keras.

In [ ]:
import os, re
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # MetaTF tools expect Keras 2

import numpy as np
import tensorflow as tf
import akida, cnn2snn, quantizeml, akida_models
from akida_models import akidanet_imagenet_pretrained
from akida_models.imagenet import get_preprocessed_samples
from cnn2snn import convert, set_akida_version, AkidaVersion

for m in (tf, akida, cnn2snn, quantizeml, akida_models):
    print(m.__name__, getattr(m, "__version__", "?"))

ALPHA, IMG_SIZE = 0.5, 224
FBZ = "akidanet_imagenet_a05_224_v1.fbz"

## 1. Pretrained quantized model (CNN baseline)

In [ ]:
# cnn2snn targets Akida 2.0 by default. The AKD1000 is Akida 1.0, so load and convert inside
# `with set_akida_version(AkidaVersion.v1):` (as in BrainChip's AkidaNet/ImageNet tutorial).
with set_akida_version(AkidaVersion.v1):
    model_keras = akidanet_imagenet_pretrained(ALPHA, quantized=True, bitwidth=4)
print("input :", model_keras.input.dtype, model_keras.input_shape)
print("output:", model_keras.output_shape)

# BrainChip's 10 copyright-free sample images (224x224x3 uint8)
x_demo, y_demo = get_preprocessed_samples(IMG_SIZE, 3)
keras_pred = np.argmax(model_keras.predict(x_demo, batch_size=100), 1)
print(f"TF-Keras accuracy on the {len(x_demo)} sample images: {np.mean(keras_pred == y_demo):.2f} (BrainChip tutorial: 9/10)")

## 2. Convert to Akida 1.0 (SNN) and save

In [ ]:
with set_akida_version(AkidaVersion.v1):
    model_akida = convert(model_keras)
print("akida IP version:", model_akida.ip_version)      # must be v1 for the AKD1000
model_akida.summary()
model_akida.save(FBZ)
print("saved", FBZ, "- load it elsewhere with akida.Model(FBZ)")

## 3. Check on the AKD1000 (needs the card)

Run on the machine with the card, with nothing else holding `/dev/akida0` (check: `fuser -v /dev/akida0`). Power measurement works on the AKD1000 only and is off by default.

In [ ]:
devices = akida.devices()
print("Available devices:", [d.desc for d in devices])
device = devices[0] if devices else None
if device is None:
    print("NO AKIDA DEVICE: steps 1-2 are done, the card check is skipped (no simulator numbers).")
else:
    assert device.version == akida.NSoC_v2, "This needs the AKD1000 (NSoC_v2)."
    model_akida.map(device)
    model_akida.summary()                       # NP allocation on the real chip

In [ ]:
if device is not None:
    acc_eval = model_akida.evaluate(x_demo, y_demo)
    card_pred = np.asarray(model_akida.predict(x_demo)).reshape(len(x_demo), -1).argmax(1)
    fwd_pred = np.asarray(model_akida.forward(x_demo)).reshape(len(x_demo), -1).argmax(1)
    print(f"AKD1000 evaluate accuracy: {acc_eval:.2f} (BrainChip tutorial: 8/10)")
    print(f"predict-argmax: accuracy {np.mean(card_pred == y_demo):.2f}, agreement with Keras {np.mean(card_pred == keras_pred):.2f}")
    print(f"forward-argmax (wrong, raw potentials): accuracy {np.mean(fwd_pred == y_demo):.2f}, "
          f"agreement with Keras {np.mean(fwd_pred == keras_pred):.2f}")

    last = model_akida.layers[-1]
    act_step = np.asarray(last.variables["act_step"])
    print(f"classifier act_step: {act_step.size} values, {len(np.unique(act_step))} distinct, "
          f"{act_step.min():.1f} to {act_step.max():.1f} (per-class scale -> use predict)")

In [ ]:
if device is not None:
    device.soc.power_measurement_enabled = True
    _ = model_akida.predict(x_demo)
    print(f"Floor power: {device.soc.power_meter.floor:.2f} mW")
    print(model_akida.statistics)               # includes fps and 'Last inference energy consumed (mJ/frame)'